# Faruq-v3 — ontology-marginal screening seed 42

Menggunakan checkpoint D0 lama dan hanya melatih C0 serta S0. Seluruh checkpoint langsung disimpan pada satu shared project folder. Test tetap terkunci.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import tarfile
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-yolo26n-baseline-v1/structured_target_audit/structured_target_support.json',
    'experiments/faruq-v3-yolo26n-baseline-v1/ontology_marginal/static_audit.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
BASELINE_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-yolo26n-baseline-v1'
SUPPORT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/structured_target_audit/structured_target_support.json')
STATIC = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/ontology_marginal/static_audit.json')
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-ontology-marginal-v1'
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT / 'faruq_grouped_summary.json').is_file():
    print('RESTORE FARUQ-V3...')
    with tarfile.open(ARCHIVE, 'r') as archive: archive.extractall('/content', filter='data')
GROUPED = DATA_ROOT / 'faruq_grouped_summary.json'
assert GROUPED.is_file()
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('PROJECT :', PROJECT_ROOT)
print('DATA    :', DATA_ROOT)
print('OUTPUT  :', OUTPUT_ROOT)

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_ontology_marginal',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED),
    '--support-report', str(SUPPORT),
    '--static-audit', str(STATIC),
    '--baseline-root', str(BASELINE_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', '42', '--device', '0',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=REPO)
started = time.monotonic()
while process.poll() is None:
    status = []
    for code in ('C0', 'S0'):
        csv_path = OUTPUT_ROOT / f'{code}_seed42/results.csv'
        epochs = max(0, len(csv_path.read_text().splitlines()) - 1) if csv_path.is_file() else 0
        last_pt = OUTPUT_ROOT / f'{code}_seed42/weights/last.pt'
        status.append(f'{code}={epochs}/50 checkpoint={last_pt.is_file()}')
    print(f'[ONTOLOGY {(time.monotonic()-started)/60:.1f} menit] ' + ', '.join(status), flush=True)
    time.sleep(60)
return_code = process.wait()
assert return_code == 0, f'Screening gagal dengan return code {return_code}. Rerun notebook untuk resume.'

In [ ]:
import json, pandas as pd
from IPython.display import display
SUMMARY = OUTPUT_ROOT / 'val_reports/screening_seed42.json'
assert SUMMARY.is_file(), SUMMARY
report = json.loads(SUMMARY.read_text())
display(pd.DataFrame([{
    'model': code, **metrics
} for code, metrics in report['models'].items()]).style.format({
    'macro_map50_95': '{:.2%}', 'bottom3_class_map50_95': '{:.2%}',
    'worst_class_map50_95': '{:.2%}', 'proposal_accessibility': '{:.2%}',
    'conditional_top1_accuracy': '{:.2%}',
}))
print('COMPARISONS:', json.dumps(report['comparisons'], indent=2))
print('FINAL:', report['decision'])
print('TEST OPENED:', report['test_opened'])
print('SUMMARY:', SUMMARY)
print('Kirim tabel dan comparisons. Jangan menjalankan seed tambahan atau membuka test.')